In [ ]:
%load_ext cudf.pandas

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

In [ ]:
# %load_ext cudf.pandas

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("/home/ElasticNotebook"))
%load_ext ElasticNotebook

# Data Exploration with Pandas
In this notebook we ll try to find insight from data on births in the United States, provided by the Centers for Disease Control (CDC). We analyze data with various methods of ***python***'s most popular library ***pandas***. To visualize the analysis we sometime apply another two popular library *matplotlib* and *seaborn*. Let's get started.

In [ ]:
# Importing all necessary libraries

import numpy as np
import pandas as pd
import time

In [ ]:
start_time = time.time()

In [ ]:
%%time
### cell 0 ###

# Alternatively you can use this code to get the dataset
# url = "https://raw.githubusercontent.com/jakevdp/data-CDCbirths/master/births.csv"
birth_data = pd.read_csv("/home/dias-benchmarks/new_notebooks/us-birth/births.csv")
birth_data.sample(5, random_state=0)

If you are seeing df.sample() first time, I tell you df.sample(5) as opposed to df.head(5) selects five random rows, thus giving you a more unbiased view of the data. The sample() method returns 1 row if a number is not specified. The column names will also be returned, in addition to the sample rows. In that case we have to add axis = 1 like df.sample(axis = 1) returns any random column.

In [ ]:
%%time
### cell 1 ###

factor = 500
# replicate the entire DataFrame in blocks via one GPU gather
# build a single index array [0,1,…,n-1,0,1,…,n-1,…] and .take it
birth_data = birth_data.take(np.tile(np.arange(len(birth_data)), factor))

# Data Exploration

In [ ]:
%%time
### cell 2 ###

birth_data.shape, birth_data.info()

We see the day column is representing days for every month. It's better to come with integer data than float. So we change here it's data type into int64. See the following results. 

In [ ]:
%%time
### cell 3 ###

birth_data = birth_data.fillna(0)
birth_data["day"] = birth_data["day"].astype(int)
birth_data.info()

In [ ]:
%%time
### cell 4 ###

birth_data.sample(5, random_state=0)

**Adding an extra column:** As the years are repeatedly there in 'year' column, to make the data analysis easy and insight of the data look natural let's add a decade column. By this all the records with year = [1960, 1961, ... , 1969] goes to decade - 1960, simillary, all rows with year = [1970, 1971, ... , 1979] goes to decade 1970 and so on. Later we see it's advantage.

In [ ]:
%%time
### cell 5 ###

# Code to add the decade column
birth_data["decade"] = 10 * (birth_data["year"] // 10)
birth_data.head()

In [ ]:
%%time
### cell 6 ###

# take a look at male and female births as a function of decade
birth_data.groupby(["decade", "gender"])["births"].sum().unstack(fill_value=0)

We immediately see that male births outnumber female births in every decade. To see this trend a bit more clearly, we can use the built-in plotting tools in Pandas to visualize the total number of births by year.

In [10]:
# %matplotlib inline
# import matplotlib.pyplot as plt
# plt.rcParams["figure.figsize"] = (10,6)
# sns.set()  # use Seaborn styles
# birth_data.pivot_table('births', index='year', columns='gender', aggfunc='sum').plot()
# plt.ylabel('total births per year')
# plt.title("Distribution Of Births Over Decades")

With a simple pivot table and built-in plot() method, we can immediately see the annual trend in births by gender. By eye, it appears that over the past 40 years male births have outnumbered female births by around 5%.

At this point anyone can make conclusion about the final trend of births. But this figure is not telling everything. Let us go into deep to explore further. Look at the code/output of two cells below.

In [11]:
# %matplotlib inline
# #import matplotlib.pyplot as plt
# plt.rcParams["figure.figsize"] = (10,6)
# sns.set()  # use Seaborn styles
# birth_data.pivot_table('births', index='year', columns='gender', aggfunc='mean').plot()
# plt.ylabel('Mean births per year')
# plt.title("Distribution Of Mean Births Over Decades")

What do you say now. There is drastic increase in mean birth rate when we move from eighties towards ninties. This was unimaginable from the previous figure where aggregator function was sum(). Was it realy happened? Look at the numerical table

In [ ]:
%%time
### cell 7 ###

# Code to present male and female mean(births) against decade numericaly
birth_data.groupby(["decade", "gender"])["births"].mean().unstack("gender")

Look at the last two rows in the pivot table above. Values are arround 30 times higher than previous values. Was it real or there is some irregularities in the dataset? We see few more charts/tables.



In [13]:
# #Boxplot for births
# #% matplotlib inline
# #import matplotlib.pyplot as plt
# sns.boxplot(y = 'births', data = birth_data)
# plt.title('Box plot - Total births')

No sufficient information gain from this figure, it seems a lot of outliers are there and maximum of them towards upper side that is year with high total births. Let's customize this box plot over decade. It may help us.

In [14]:
# #boxplot of births by decade
# fig, ax = plt.subplots(figsize=(10, 8))
# sns.boxplot(x = 'decade', y = 'births', data = birth_data, ax = ax)
# plt.title('Box plot - Total births over decade')
# plt.show()

Now it's clear they are not all outliers. Actualy change of distribution is happening when we enter nineties.

In [ ]:
%%time
### cell 8 ###

years2check = [1988, 1989]
# Single GPU-accelerated filter instead of scanning twice
filtered = birth_data[birth_data["year"].isin(years2check)]
# Group on GPU and iterate only over the filtered years
for year, yearly_data in filtered.groupby("year"):
    print("Births data for -", year)
    print(yearly_data)

**Note:** Two major changes in data collection we observeed. First: from year 1969 to 1988 data collected/stored day wise for male and female separately. But from year 1989 onwards data collected monthwise for male and female separately i.e. for every year in [1989, 1990, ... , 2008] there are 24 records are there of which 12 for male and remaining 12 for female. Second: the day column has value day wise i.e. from [1, 2, ... , 31] till year 1988  with extra two column with value 99 for each male and female while from year 1989 onwards day column has value zero (initial NAN value, we changed to zero). That is why there are 480 = 24 * 20 zero values there, from 1989 to 2008 is 20 years.

Change in data collection started from 1989. Now if you look again the above box plot we understand that it'd be better we analyze data separately. Let's get into the code.

In [ ]:
%%time
### cell 9 ###

# Descriptive summary before removing outliers
birth_data.describe()

In [ ]:
%%time
### cell 10 ###

# Compute quartiles on GPU
quartiles = birth_data["births"].quantile([0.25, 0.5, 0.75])
q25, mu, q75 = quartiles.iloc[0], quartiles.iloc[1], quartiles.iloc[2]

# Sigma and bounds (still small Python scalars)
sig = 0.74 * (q75 - q25)
lower_bound = mu - 5 * sig
upper_bound = mu + 5 * sig

# Filter on GPU via boolean indexing
births = birth_data[
    (birth_data["births"] > lower_bound) & (birth_data["births"] < upper_bound)
]

birth_data.shape, births.sample(5, random_state=0)

In [15]:
# #Code to remove outliers or actualy to consider one part of the data
# quartiles = np.percentile(birth_data['births'], [25, 50, 75])
# mu = quartiles[1]
# sig = 0.74 * (quartiles[2] - quartiles[0])
# births = birth_data.query('(births > mu - 5 * sig) & (births < mu + 5 * sig)')
# print(births.shape)
# births.sample(5)

In [ ]:
%%time
### cell 11 ###

# Descriptive summary after removal of outliers
births.describe()

If we compare these two summaries we observe the major difference is in births column. Before mean, std = 9762.3, 28552.5; after mean, std = 4824.5, 580. One more thing to say: this new data births(saved after removing outliers) must have years up to 1988. Let's run the following code.

In [ ]:
%%time
### cell 12 ###

births.year.unique()

Finally, we can combine the day, month, and year to create a Date index. This allows us to quickly compute the weekday corresponding to each row. Think about the records after 1988 where day val is zero for all rows. So we could not make this action.

In [ ]:
%%time
### cell 13 ###

# create a datetime index from the year, month, day
births.index = pd.to_datetime(
    10000 * births.year + 100 * births.month + births.day, format="%Y%m%d"
)
births.head()

In [ ]:
%%time
### cell 14 ###

# Creating another column 'dayofweek' using dayofweek attribute of pandas DatetimeIndex
births["dayofweek"] = births.index.dayofweek
births.head()

In [ ]:
%%time
### cell 15 ###

mapping = {
    0: "Monday",
    1: "Tuesady",
    2: "Wednesday",
    3: "Thursday",
    4: "Friday",
    5: "Saturday",
    6: "Sunday",
}
births["dayofweek"] = births["dayofweek"].map(mapping)

In [ ]:
%%time
### cell 16 ###

births.head()

In [22]:
# # plot for births by week days over decades
# %matplotlib inline
# import matplotlib.pyplot as plt
# plt.rcParams["figure.figsize"] = (10,6)
# sns.set()  # use Seaborn styles
# births.pivot_table('births', index='dayofweek', columns='decade', aggfunc='mean').plot()
# plt.ylabel('mean births by day')
# plt.title("Trend Of Mean Births Along Week Days Over Decades ")

What do we conclude here? Births are slightly less common on weekends than on weekdays! It is common since decades. Note that unless we removed 1990s and 2000s decades we could not have this plot because the CDC data contains only the month of birth starting from 1989.

Our last attempt is to view the plot as the mean number of births by the day of the year over decades. Let's first group the data by month and day separately. so that dataframe will have 366 (+1 for leap year) records where first value represents the mean births of all births on 1st Jan from 1969 to 1988.

In [ ]:
%%time
### cell 17 ###

# Optimized for cudf: use GPU groupby instead of pivot_table
births_by_date = (
    births[["births"]].groupby([births.index.month, births.index.day]).mean()
)
births_by_date.sample(10, random_state=0)

The first row above interpretes that mean births on 1st October from 1969 to 1988 is 5167.325 and the same way other rows can be interpreted. What we created above is a multi index dataframe which have total 366 rows.

In [ ]:
%%time
### cell 18 ###

births_by_date.shape

To plot this data easily we turn these month-day multi index into a date (like time series) by associating them with a dummy year variable (say) 2020. In fact you can take any leap year.

In [ ]:
%%time
### cell 19 ###

# Vectorized conversion to datetime index using GPU-accelerated to_datetime
births_by_date.index = pd.to_datetime(
    {
        "year": 2020,
        "month": births_by_date.index.get_level_values(0),
        "day": births_by_date.index.get_level_values(1),
    }
)
births_by_date.head()

In [ ]:
%%time
### cell 20 ###

end_time = time.time()
print(end_time - start_time)

So we have a dataframe (like time series) with only values the average number of births by date of the year. Now we can use the simple DataFrame.plot() method to plot the data. let's see the code/output.

In [30]:
# # Plot the results
# fig, ax = plt.subplots(figsize=(12, 6))
# births_by_date.plot(ax=ax)